In [ ]:
!pip install transformers datasets seqeval torch

In [ ]:
# STEP 1: Imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate


In [ ]:
# STEP 2: Load Dataset
dataset = load_dataset("wikiann", "en")

In [ ]:
# 🔥 Use more data (better learning)
dataset["train"] = dataset["train"].select(range(20000))
dataset["validation"] = dataset["validation"].select(range(2000))

In [ ]:
# STEP 3: Labels
label_list = dataset["train"].features["ner_tags"].feature.names

In [ ]:
# STEP 4: Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# STEP 5: Tokenization + Label Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True,
        max_length=128
    )

    labels = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        word_labels = examples["ner_tags"][i]

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
# STEP 6: Model (🔥 Use BERT for better accuracy)
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

In [ ]:
# STEP 7: Training Args (balanced speed + accuracy)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,   # 🔥 increased
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

In [ ]:
# STEP 8: Metrics
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        curr_preds = []
        curr_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                curr_preds.append(label_list[p_val])
                curr_labels.append(label_list[l_val])

        true_predictions.append(curr_preds)
        true_labels.append(curr_labels)

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [ ]:
# STEP 9: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

In [ ]:
# STEP 10: Train the model
trainer.train()

In [ ]:
# STEP 11: Evaluate the model
print(trainer.evaluate())

In [ ]:
# STEP 12: 🔥 CLEAN INFERENCE (pipeline)
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


In [ ]:
def format_output(sentence, results):
    print("Input Sentence:")
    print(sentence)
    print("\nModel Predictions:")
    print("-----------------------------------")

    for item in results:
        word = item['word']
        entity = item['entity_group']
        score = round(float(item['score']), 2)

        if entity == "PER":
            label = "Person"
        elif entity == "ORG":
            label = "Organization"
        elif entity == "LOC":
            label = "Location"
        else:
            label = entity

        print(f"{word.capitalize():<12} → {label} (Confidence: {score})")

    print("-----------------------------------")

In [ ]:
sentence = "Elon_Musk founded Tesla in California"

results = ner_pipeline(sentence)

format_output(sentence, results)